# Distri Days SDK Example

In this example, the python SDK is used to take a snapshot.

Prerequirements:
- Cuvis installed
- Calibration File in ./factory
- Camera connected

In [ ]:
import cuvis
import time

print("Cuvis Python SDK Example 1")
print(cuvis.version())

In [ ]:
import os

snapshot_integration_time_ms = 100
save_directory = "./output"

camera_serial_number_str = "Your camera serial here"

camera_calibration_file_path = F"./factory/{camera_serial_number_str}.cu3c"

## Calibration File (.cu3c)

This calibration file contains everything needed to connect to and process data from your camera.
The **.cu3c** camera calibration file format is a special form of the SessionFile format **.cu3s**

It is thus used as a calibration file and usually contains (among other things):
 - The actual encrypted camera calibration file
 - A spectral radiance calibration file
 - Test references (Dark and White)
 - A standard camera recording configuration (framerate, integration time, etc.)

In [ ]:
print("Load camera calibration file")
calib = cuvis.SessionFile(camera_calibration_file_path)

## Acquisition Context

The *Acquisition Context* is your interface to control the camera and all aspects of the data acquisition.

Initialize it using a *SessionFile* object, then set the recording parameters and start an acquisition.
As soon as the **AcquisitionContext** is created, it will try to establish a connection with the camera.

Here, the "Software" operation mode is used to enable data acquisition using a software trigger.
This is also called snapshot mode.

*Please note:*
The *AcquisitionContext* will **only** connect to the **exact** camera of the same serial number matching the calibration file!
All other cameras/devices are ignored.

In [ ]:
print("Loading Acquisition Context")
acq = cuvis.AcquisitionContext(calib)

print("Connecting with camera")
while(not acq.ready):
    time.sleep(1)
    print(".", end="")
print("\nCamera connected!")
time.sleep(0.5)

# Set camera to software trigger
acq.operation_mode = cuvis.OperationMode.Software
acq.integration_time = snapshot_integration_time_ms

## Capture Measurement with Software Trigger

Using the `capture()` method, a single measurement is initiated.
Taking a snapshot requires some time, so, to prevent the call to `capture()` from blocking execution, an *AsyncMesu* is returned.
To await the completion of the snapshot, use the `get()` method on the *AsyncMesu*.

In [ ]:
# Optional: Name the recording
acq.session_info = cuvis.SessionData("My_Measurement", 0, 0)

# With the camera connection established, a measurement can be triggered using the capture() method.
# This returns an AsyncMesu object
async_mesu = acq.capture()

# To get the actual measurement, wait on the AsyncMesu using the get() method.
mesu, status = async_mesu.get(timeout_ms=5000)
print(F"Measurement reports: {status}")

## Export Measurement

To save measurements in the Cubert file format *SessionFile* (.cu3s), use the *CubeExporter*.
Using *SessionFiles* to save measurements, is highly recommended, as it can store multiple raw measurements, reference measurements and meta-data
all in one file, allowing you to re-process the measurements after the fact.
This enables you to fine-tune the final product, convert the output format and even fix some mistakes made during data acquisition.

In [ ]:
save_config = cuvis.FileWriteSettings.SaveArgs(export_dir=save_directory)
exporter = cuvis.CubeExporter(save_config)

# Export the measurement - write the data to the disk in SessionFile format using the CubeExporter
if status == cuvis.Async.AsyncResult.done:
    exporter.apply(mesu)
    print("Measurement exported!")
else:
    print("Something went wrong!")